# Import libraries

In [46]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from googleapiclient.discovery import build
from transformers import pipeline
from dotenv import load_dotenv
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta
from atproto import Client
import time
from langdetect import detect, LangDetectException
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import re

# Load data

In [39]:
# Load environment variables
load_dotenv()
BSKY_HANDLE = os.getenv('BSKY_HANDLE')
BSKY_PASSWORD = os.getenv('BSKY_PASSWORD')

# Keep exactly the same timeline as the YouTube analysis
SEARCH_QUERY = ['AI', 'Artificial Intelligence', 'AI Bubble', 'ChatGPT']
START_DATE = datetime(2022, 1, 1, tzinfo=timezone.utc)
END_DATE = datetime(2025, 12, 31, tzinfo=timezone.utc)
POSTS_PER_MONTH = 100

# Authenticate
client = Client()
client.login(BSKY_HANDLE, BSKY_PASSWORD)
print("Connected to Bluesky successfully!")

Connected to Bluesky successfully!


In [40]:
def is_english(text):
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False

In [41]:
all_bsky_data = []
current_date = START_DATE

print(f"Starting temporal Bluesky analysis from {START_DATE.year} to {END_DATE.year}...")

all_bsky_data = []
seen_posts = set()

current_date = START_DATE

while current_date < END_DATE:
    next_month = current_date + relativedelta(months=1)
    month_label = current_date.strftime("%Y-%m")

    print(f"Processing Bluesky posts for {month_label}...")

    for query in SEARCH_QUERY:
        
        date_query = f'{query} since:{current_date.strftime("%Y-%m-%dT%H:%M:%SZ")} until:{next_month.strftime("%Y-%m-%dT%H:%M:%SZ")}'  
       
        try:
            response = client.app.bsky.feed.search_posts({
                "q": date_query,
                "limit": POSTS_PER_MONTH,
                "sort": 'top'
            })

            for post in response.posts:
                post_id = post.uri

                safe_text = post.record.text[:500] if hasattr(post.record, 'text') else ""
                if post_id in seen_posts or not is_english(safe_text):
                    continue

                seen_posts.add(post_id)

                text = post.record.text

                likes = getattr(post, "like_count", 0) or 0
                reposts = getattr(post, "repost_count", 0) or 0
                replies = getattr(post, "reply_count", 0) or 0

                influence_weight = 1 + likes + reposts + replies

                all_bsky_data.append({
                    "Month": month_label,
                    "Date": pd.to_datetime(post.record.created_at),
                    "Query": query,
                    "Post_ID": post_id,
                    "Text": text,
                    "Likes": likes,
                    "Reposts": reposts,
                    "Replies": replies,
                    "Weight": influence_weight
                })
        except Exception as e:
            print(f"  -> Error for {month_label}, query '{query}': {e}")
    time.sleep(0.5)
    print(f"Finished processing {month_label}. Total posts collected so far: {len(all_bsky_data)}")
    current_date = next_month

bsky_df = pd.DataFrame(all_bsky_data)
bsky_df["Month"] = pd.to_datetime(bsky_df["Month"])

print(f"\nDone! Retrieved {len(bsky_df)} unique Bluesky posts.")
display(bsky_df.head())

Starting temporal Bluesky analysis from 2022 to 2025...
Processing Bluesky posts for 2022-01...
Finished processing 2022-01. Total posts collected so far: 56
Processing Bluesky posts for 2022-02...
Finished processing 2022-02. Total posts collected so far: 110
Processing Bluesky posts for 2022-03...
Finished processing 2022-03. Total posts collected so far: 183
Processing Bluesky posts for 2022-04...
Finished processing 2022-04. Total posts collected so far: 245
Processing Bluesky posts for 2022-05...
Finished processing 2022-05. Total posts collected so far: 312
Processing Bluesky posts for 2022-06...
Finished processing 2022-06. Total posts collected so far: 403
Processing Bluesky posts for 2022-07...
Finished processing 2022-07. Total posts collected so far: 472
Processing Bluesky posts for 2022-08...
Finished processing 2022-08. Total posts collected so far: 554
Processing Bluesky posts for 2022-09...
Finished processing 2022-09. Total posts collected so far: 626
Processing Bluesky

,Month,Date,Query,Post_ID,Text,Likes,Reposts,Replies,Weight
0,2022-01-01,2022-01-13 14:50:33+00:00,AI,at://did:plc:xy5jxwppps74gfzj5h7tpibz/app.bsky...,This AI-inserted referral-money-making panel u...,0,0,0,1
1,2022-01-01,2022-01-04 03:55:16+00:00,AI,at://did:plc:uxkyer4ru2tkxirh4ajyl3a4/app.bsky...,the #Data Science Daily is out! #bigdata #data...,0,0,0,1
2,2022-01-01,2022-01-20 08:48:01+00:00,AI,at://did:plc:zopuy2ivmbsygcb27y7z5bhn/app.bsky...,"Save the Date: May 09-11, 2022 - and check out...",0,0,0,1
3,2022-01-01,2022-01-03 01:21:24+00:00,AI,at://did:plc:hmtimeybcvsw23mfeqa6g7fs/app.bsky...,I've just updated my webpage with some great a...,0,0,0,1
4,2022-01-01,2022-01-28 02:31:03+00:00,AI,at://did:plc:ujdxpw74vjocirexh7pjfbdt/app.bsky...,Accenture CEO Julie Sweet asserts that we'll s...,1,0,0,2


In [42]:
print(f"\nDone! Retrieved {len(bsky_df)} unique Bluesky posts.")
display(bsky_df['Query'].value_counts())


Done! Retrieved 10149 unique Bluesky posts.


Query
Artificial Intelligence    3312
AI Bubble                  2458
AI                         2301
ChatGPT                    2078
Name: count, dtype: int64

In [44]:
# Save to CSV
bsky_df.to_csv("bluesky_ai_posts.csv", index=False)